[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/00_setup_verification.ipynb)

# 00 — Verificación del Entorno

Este notebook verifica que todas las dependencias, datos y módulos estén correctamente configurados.

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
DATA_ROOT_OVERRIDE = None

if IN_COLAB:
    # ── Descargar dataset desde Kaggle ────────────────────────────────────
    import subprocess
    subprocess.run(["pip", "install", "kaggle", "-q"], check=True)

    from google.colab import files
    print("Sube tu archivo kaggle.json (descárgalo en kaggle.com → Settings → API):")
    files.upload()

    os.makedirs('/root/.kaggle', exist_ok=True)
    if os.path.exists('kaggle.json'):
        os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
        os.chmod('/root/.kaggle/kaggle.json', 0o600)
        print("✓ Credenciales configuradas")

    print("Descargando dataset Landslide4Sense...")
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", "lxrzp/landslide4sense",
         "-p", "/content/", "--unzip"],
        check=True
    )

    # Buscar dónde quedó TrainData
    for _root, _dirs, _ in os.walk('/content'):
        if 'TrainData' in _dirs:
            DATA_ROOT_OVERRIDE = _root
            print(f"✓ Dataset encontrado en: {DATA_ROOT_OVERRIDE}")
            break
        _depth = _root.replace('/content', '').count(os.sep)
        if _depth >= 4:
            _dirs.clear()

    if not DATA_ROOT_OVERRIDE:
        print("No se encontró TrainData automáticamente.")
        print("Ajusta DATA_ROOT_OVERRIDE manualmente en la siguiente celda.")
else:
    print('Entorno local — se usará detección automática de rutas.')


In [ ]:
import os, sys
from pathlib import Path

# Preservar DATA_ROOT_OVERRIDE si ya fue fijada por la celda de Kaggle
try:
    DATA_ROOT_OVERRIDE
except NameError:
    DATA_ROOT_OVERRIDE = None

# ── Detección automática ──────────────────────────────────────────────────────
if DATA_ROOT_OVERRIDE:
    DATA_ROOT = Path(DATA_ROOT_OVERRIDE)
    print(f"Usando ruta configurada: {DATA_ROOT}")
else:
    _cwd = Path(os.getcwd())
    _candidates = [
        _cwd / ".." / "data",
        _cwd / "..",
        _cwd / "data",
        _cwd,
        Path("/content/landslide4sense"),
        Path("/content/data"),
        Path("/content"),
    ]
    DATA_ROOT = None
    for _c in _candidates:
        if (_c / "TrainData" / "img").exists():
            DATA_ROOT = _c.resolve()
            print(f"Dataset detectado automáticamente en: {DATA_ROOT}")
            break

if DATA_ROOT is None or not (DATA_ROOT / "TrainData" / "img").exists():
    raise FileNotFoundError(
        "Dataset no encontrado.\n"
        "Verifica que la celda de Kaggle corrió correctamente,\n"
        "o ajusta DATA_ROOT_OVERRIDE manualmente."
    )

# ── Verificar estructura ──────────────────────────────────────────────────────
partitions = {
    "TrainData": {"img": True,  "mask": True},
    "ValidData": {"img": True,  "mask": False},
    "TestData":  {"img": True,  "mask": False},
}

all_ok = True
for part, dirs in partitions.items():
    for subdir, required in dirs.items():
        p = DATA_ROOT / part / subdir
        n = len(list(p.glob("*.h5"))) if p.exists() else -1
        status = "✓" if n > 0 else ("✗" if required else "—")
        print(f"  {status}  {part}/{subdir:<10} → {n if n >= 0 else 'no existe':>5} archivos .h5")
        if required and n <= 0:
            all_ok = False

if not all_ok:
    print("\n⚠️  Dataset incompleto.")
else:
    print("\n✓ Dataset correctamente estructurado.")


## 1. Instalación de dependencias

In [ ]:
import subprocess, sys

packages = [
    "torch torchvision",
    "timm",
    "segmentation-models-pytorch",
    "h5py",
    "scikit-learn",
    "matplotlib seaborn",
    "pyyaml",
]

IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    for pkg in packages:
        print(f"Instalando {pkg}...")
        subprocess.run([sys.executable, "-m", "pip", "install"] + pkg.split() + ["-q"], check=True)
    print("✓ Todas las dependencias instaladas")
else:
    print("Entorno local — verifica que el entorno conda/venv esté activo")


## 2. Verificación de imports

In [ ]:
import torch
import h5py
import numpy as np
import sklearn
import timm
import segmentation_models_pytorch as smp

print(f"torch     : {torch.__version__}")
print(f"CUDA      : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
print(f"h5py      : {h5py.__version__}")
print(f"numpy     : {np.__version__}")
print(f"sklearn   : {sklearn.__version__}")
print(f"timm      : {timm.__version__}")
print(f"smp       : {smp.__version__}")
print("\n✓ Todos los imports correctos")


## 3. Visualización de un patch de muestra

In [ ]:
import h5py, numpy as np, matplotlib.pyplot as plt

img_files = sorted((DATA_ROOT / "TrainData" / "img").glob("*.h5"))
print(f"Total patches: {len(img_files)}")

with h5py.File(img_files[0], 'r') as f:
    patch = f['img'][:]

print(f"Patch shape : {patch.shape}")
print(f"Dtype       : {patch.dtype}")
print(f"Min / Max   : {patch.min():.3f} / {patch.max():.3f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, ch, name in zip(axes, [0, 7, 9], ['Ch0 (Red)', 'Ch7 (SAR VV)', 'Ch9 (DEM)']):
    ax.imshow(patch[:, :, ch], cmap='gray')
    ax.set_title(name)
    ax.axis('off')
plt.suptitle(f"Patch: {img_files[0].name}")
plt.tight_layout()
plt.show()


## 4. Test rápido del módulo `src/`

In [ ]:
import sys
sys.path.insert(0, "..")

from src.config import get_debug_config
from src.dataset import Landslide4SenseDataset
from src.models import build_model, model_summary
from src.utils import set_seed, get_device

cfg = get_debug_config()
cfg.data_root = str(DATA_ROOT)

set_seed(cfg.seed)
device = get_device()
print(f"Dispositivo: {device}")

model = build_model("resnet50", n_channels=14, pretrained=False)
print(model_summary(model, n_channels=14))

ds = Landslide4SenseDataset(
    img_dir=str(DATA_ROOT / "TrainData" / "img"),
    mask_dir=str(DATA_ROOT / "TrainData" / "mask"),
    indices=list(range(8)),
    normalize=True,
)
sample = ds[0]
print(f"\nBatch de prueba:")
print(f"  image shape: {sample['image'].shape}")
print(f"  label      : {sample['label'].item()}")
print(f"  filename   : {sample['filename']}")
print("\n✓ Módulo src/ funcionando correctamente.")


## Resumen

Si todas las celdas se ejecutaron sin errores, el entorno está listo para:

1. `01_eda_analysis.ipynb` — Análisis exploratorio de datos
2. `02_preprocessing.ipynb` — Preprocesamiento y augmentación
3. `03_baseline_rf.ipynb` — Baseline Random Forest
4. `04_resnet50.ipynb` — ResNet-50 (14 canales)
5. `05_efficientnet_b4.ipynb` — EfficientNet-B4
6. `06_unet_segmentation.ipynb` — U-Net + ResNet-34
7. `07_evaluation_comparison.ipynb` — Comparación de modelos